# NexusBots — Enhanced Prompt Benchmark

**Goal:** Evaluate `Qwen3-0.6B` BASE model on our 100-row test set under two conditions:

| System | Description |
|---|---|
| `BASE + Original Prompt` | Simple tool-list prompt (baseline ~0.60) |
| `BASE + Enhanced Prompt` | Decision tree + 8 contrastive few-shots (target ~0.79) |

**Metrics:** Tool Accuracy · Arg F1 · UI Guide Accuracy · p50 Latency (ms) · Per-language breakdown

**Dataset:** `finetune/data/test.jsonl` — 100 held-out rows, never used in training

## Cell 1 — Install dependencies (run once)

In [2]:
# Run this cell only if packages are not already installed
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch", "transformers>=4.41", "accelerate", "matplotlib", "pandas", "tabulate")
print("All packages ready.")

All packages ready.


## Cell 2 — Imports & paths

In [3]:
import json, re, sys, time, warnings
from collections import defaultdict
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = Path(".").resolve()                          # finetune/eval/
FINETUNE_ROOT = NOTEBOOK_DIR.parent                          # finetune/
PROJECT_ROOT  = FINETUNE_ROOT.parent                         # NexusBots-TDL-Project/

TEST_PATH     = FINETUNE_ROOT / "data" / "test.jsonl"
RESULTS_DIR   = PROJECT_ROOT / "research" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Add finetune/ to path so we can import config + enhanced_prompt
sys.path.insert(0, str(FINETUNE_ROOT))

assert TEST_PATH.exists(), f"Test file not found: {TEST_PATH}"
print(f"Project root : {PROJECT_ROOT}")
print(f"Test set     : {TEST_PATH}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    free_gb = torch.cuda.mem_get_info(0)[0] / 1024**3
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM          : {free_gb:.1f} GB free / {total_gb:.1f} GB total")

/home/cs412/Downloads/Subjects/TDL/tdl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root : /home/cs412/Downloads/Subjects/TDL/NexusBots-TDL-Project
Test set     : /home/cs412/Downloads/Subjects/TDL/NexusBots-TDL-Project/finetune/data/test.jsonl
Results dir  : /home/cs412/Downloads/Subjects/TDL/NexusBots-TDL-Project/research/results
CUDA available: True
GPU           : Quadro P400
VRAM          : 1.1 GB free / 1.9 GB total


## Cell 3 — Load system prompts

In [4]:
from config import SYSTEM_PROMPT, BASE_MODEL
from eval.enhanced_prompt import ENHANCED_SYSTEM_PROMPT

ORIGINAL_PROMPT  = SYSTEM_PROMPT          # from config.py
ENHANCED_PROMPT  = ENHANCED_SYSTEM_PROMPT # from eval/enhanced_prompt.py

print(f"Base model      : {BASE_MODEL}")
print(f"Original prompt : {len(ORIGINAL_PROMPT):,} chars")
print(f"Enhanced prompt : {len(ENHANCED_PROMPT):,} chars")
print()
print("─" * 60)
print("[ENHANCED PROMPT PREVIEW — first 800 chars]")
print("─" * 60)
print(ENHANCED_PROMPT[:800], "...")

Base model      : Qwen/Qwen3-0.6B
Original prompt : 4,295 chars
Enhanced prompt : 5,522 chars

────────────────────────────────────────────────────────────
[ENHANCED PROMPT PREVIEW — first 800 chars]
────────────────────────────────────────────────────────────
You are a robotics e-commerce function router. Output ONE JSON object: {"tool":"...","arguments":{...},"ui_guide":"..."|null}

Products (id|name|category|price):
1|Amazon Astro|Kitchen|$1599
2|Samsung Ballie|Kitchen|$1299
3|Enabot EBO X|Kitchen|$599
4|iRobot Roomba j9+|Home Cleaner|$799
5|Roborock S8 MaxV Ultra|Home Cleaner|$1799
6|Ecovacs WINBOT W2 Omni|Home Cleaner|$499
7|Ring Always Home Cam|Drone|$249
8|DJI Matrice 30T|Drone|$13600
9|Aiper Surfer S1|Drone|$1399
10|Miko 3|Humanoid|$249
11|Wonder Workshop Dash|Humanoid|$149
12|LEGO Education Spike Prime|Humanoid|$395

Tools and required arguments:
- search_products: query (string), category? (Kitchen|Home Cleaner|Drone|Humanoid)
- get_product: product_id (int 1-12)
- compare_pr

## Cell 4 — Load test dataset

In [5]:
rows = [json.loads(line) for line in TEST_PATH.open() if line.strip()]
print(f"Loaded {len(rows)} test rows")

# Quick dataset summary
tools    = defaultdict(int)
langs    = defaultdict(int)
profils  = defaultdict(int)

for r in rows:
    tools[r.get("tool_name", "unknown")] += 1
    langs[r.get("language",  "unknown")] += 1
    profils[r.get("proficiency", "unknown")] += 1

print("\nTool distribution:")
for t, c in sorted(tools.items(), key=lambda x: -x[1]):
    print(f"  {t:<25} {c:>3}  {'█' * c}")

print("\nLanguage distribution:")
for l, c in sorted(langs.items()):
    print(f"  {l:<6} {c:>3}")

print("\nProficiency distribution:")
for p, c in sorted(profils.items()):
    print(f"  {p:<10} {c:>3}")

Loaded 100 test rows

Tool distribution:
  navigate_to                27  ███████████████████████████
  search_products            25  █████████████████████████
  recommend                  16  ████████████████
  get_product                11  ███████████
  add_to_cart                11  ███████████
  compare_products           10  ██████████

Language distribution:
  en      55
  hi      27
  te      18

Proficiency distribution:
  beginner    56
  expert      44


## Cell 5 — Load Qwen3-0.6B model

In [6]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading tokenizer from {BASE_MODEL} ...")
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Loading model on {DEVICE} ({DTYPE}) ...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()
print(f"Model loaded in {time.time()-t0:.1f}s")

param_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Parameters    : {param_m:.0f}M")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1024**3
    print(f"VRAM used     : {used:.2f} GB")

Loading tokenizer from Qwen/Qwen3-0.6B ...
Loading model on cuda:0 (torch.float16) ...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:  57%|█████▋    | 177/311 [00:00<00:00, 382.11it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 1.95 GiB of which 42.31 MiB is free. Process 3338 has 128.94 MiB memory in use. Including non-PyTorch memory, this process has 1.09 GiB memory in use. Of the allocated memory 1.06 GiB is allocated by PyTorch, and 1.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Cell 6 — Helper functions

In [ ]:
VALID_TOOLS = {
    "search_products", "get_product", "compare_products",
    "recommend", "add_to_cart", "navigate_to",
}


def parse_json(text: str) -> dict:
    """Robustly extract JSON from model output."""
    # 1. Strip <think>...</think> blocks (Qwen3 CoT output)
    text = re.sub(r"<think>[\s\S]*?</think>", "", text).strip()
    # 2. Strip unclosed <think> blocks (truncated by token limit)
    text = re.sub(r"<think>[\s\S]*$", "", text, flags=re.DOTALL).strip()
    # 3. Strip markdown fences
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE).strip()
    # 4. Direct parse
    try:
        return json.loads(text)
    except Exception:
        pass
    # 5. Find outermost braces
    s, e = text.find("{"), text.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            return json.loads(text[s:e])
        except Exception:
            pass
    return {}


def extract_gold(row: dict):
    """Extract (tool, arguments, ui_guide) from a test row."""
    for msg in row.get("messages", []):
        if msg["role"] == "assistant":
            obj = json.loads(msg["content"])
            return obj["tool"], obj.get("arguments", {}), obj.get("ui_guide")
    return "", {}, None


def extract_query_context(row: dict):
    """Extract (query_string, context_string) from user message."""
    user_msg = next(
        (m["content"] for m in row["messages"] if m["role"] == "user"), ""
    )
    parts = user_msg.split("\nContext: ", 1)
    q   = parts[0].replace("Query: ", "").strip()
    ctx = parts[1].strip() if len(parts) > 1 else "{}"
    return q, ctx


def score_arg_f1(pred: dict, gold: dict) -> float:
    """Compute field-level F1 between predicted and gold arguments."""
    if not gold:
        return 1.0 if not pred else 0.0
    keys = set(gold) | set(pred)
    if not keys:
        return 1.0
    hits = sum(
        str(pred.get(k, "")).strip().lower() == str(gold.get(k, "")).strip().lower()
        for k in keys
    )
    return hits / len(keys)


def infer(query: str, ctx: str, sysprompt: str, max_new_tokens: int = 160):
    """Run one inference with ChatML format. Returns (parsed_dict, latency_ms)."""
    msgs = [
        {"role": "system", "content": sysprompt},
        {"role": "user",   "content": f"Query: {query}\nContext: {ctx}"},
    ]
    # enable_thinking=False → disables Qwen3 chain-of-thought
    text = tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    ids = tok(text, return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tok.eos_token_id,
        )
    lat_ms = (time.perf_counter() - t0) * 1000

    raw = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_json(raw), lat_ms, raw


print("Helper functions defined.")

## Cell 7 — Evaluation loop

In [ ]:
def evaluate(label: str, sysprompt: str, rows: list, verbose: bool = True):
    """
    Run full evaluation over test set.
    Returns:
      stats  : dict[lang] → {ok, f1, guide_ok, n, lats}
      records: list of per-row dicts (for DataFrame)
      fails  : list of failure rows
    """
    stats   = defaultdict(lambda: {"ok": 0, "f1": 0.0, "guide_ok": 0, "n": 0, "lats": []})
    records = []
    fails   = []
    n = len(rows)
    running_ok = 0

    print(f"\n{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")

    for i, row in enumerate(rows):
        gold_tool, gold_args, gold_guide = extract_gold(row)
        q, ctx = extract_query_context(row)
        lang   = row.get("language", "en")

        try:
            pred, lat_ms, raw_text = infer(q, ctx, sysprompt)
            pred_tool  = pred.get("tool", "")
            pred_args  = pred.get("arguments", {}) if isinstance(pred.get("arguments"), dict) else {}
            pred_guide = pred.get("ui_guide")
        except Exception as ex:
            pred_tool, pred_args, pred_guide, lat_ms, raw_text = "", {}, None, 0.0, str(ex)

        ok       = int(pred_tool == gold_tool)
        f1       = score_arg_f1(pred_args, gold_args)
        guide_ok = int(pred_guide == gold_guide)
        running_ok += ok

        stats[lang]["ok"]       += ok
        stats[lang]["f1"]       += f1
        stats[lang]["guide_ok"] += guide_ok
        stats[lang]["n"]        += 1
        stats[lang]["lats"].append(lat_ms)

        records.append({
            "row_id":     i + 1,
            "language":   lang,
            "proficiency":row.get("proficiency", ""),
            "query":      q[:70],
            "gold_tool":  gold_tool,
            "pred_tool":  pred_tool,
            "tool_ok":    ok,
            "arg_f1":     round(f1, 4),
            "gold_guide": gold_guide,
            "pred_guide": pred_guide,
            "guide_ok":   guide_ok,
            "lat_ms":     round(lat_ms, 1),
        })

        if not ok:
            fails.append(records[-1] | {"raw_output": raw_text[:200]})

        if verbose:
            bar = "█" * ((i + 1) * 25 // n) + "░" * (25 - (i + 1) * 25 // n)
            print(
                f"  [{bar}] {i+1:3d}/{n}  {'✓' if ok else '✗'}  "
                f"pred={pred_tool or '(none)':22s}  "
                f"gold={gold_tool:22s}  "
                f"acc={running_ok/(i+1):.3f}",
                flush=True,
            )

    final_acc = running_ok / n
    print(f"\n  ✔ {label} DONE → tool_acc = {final_acc:.4f}  ({running_ok}/{n})\n")
    return dict(stats), records, fails

## Cell 8 — Run Enhanced Prompt evaluation

In [ ]:
enh_stats, enh_records, enh_fails = evaluate(
    label     = "BASE + ENHANCED PROMPT (decision tree + 8 few-shots)",
    sysprompt = ENHANCED_PROMPT,
    rows      = rows,
)

## Cell 9 — Run Original Prompt evaluation (baseline)

In [ ]:
orig_stats, orig_records, orig_fails = evaluate(
    label     = "BASE + ORIGINAL PROMPT (baseline)",
    sysprompt = ORIGINAL_PROMPT,
    rows      = rows,
)

## Cell 10 — Aggregate results

In [ ]:
def summarise(label: str, stats: dict) -> dict:
    total_ok  = sum(v["ok"]       for v in stats.values())
    total_f1  = sum(v["f1"]       for v in stats.values())
    total_gok = sum(v["guide_ok"] for v in stats.values())
    n         = sum(v["n"]        for v in stats.values())
    lats      = sorted(l for v in stats.values() for l in v["lats"])
    p50 = lats[len(lats) // 2] if lats else 0.0
    p95 = lats[int(len(lats) * 0.95)] if lats else 0.0
    return {
        "system":        label,
        "tool_acc":      round(total_ok  / n, 4),
        "arg_f1":        round(total_f1  / n, 4),
        "ui_guide_acc":  round(total_gok / n, 4),
        "p50_ms":        round(p50, 1),
        "p95_ms":        round(p95, 1),
        "correct":       total_ok,
        "total":         n,
    }


results = [
    summarise("Qwen3-0.6B BASE + Enhanced Prompt", enh_stats),
    summarise("Qwen3-0.6B BASE + Original Prompt", orig_stats),
]

df_results = pd.DataFrame(results)

print("\n" + "="*75)
print("  FINAL RESULTS")
print("="*75)
print(df_results[["system","tool_acc","arg_f1","ui_guide_acc","p50_ms","p95_ms","correct","total"]]
      .to_string(index=False))
print()

delta_tool  = results[0]["tool_acc"] - results[1]["tool_acc"]
delta_f1    = results[0]["arg_f1"]   - results[1]["arg_f1"]
delta_guide = results[0]["ui_guide_acc"] - results[1]["ui_guide_acc"]
print(f"  Δ Tool Acc    (Enhanced - Original) : {delta_tool:+.4f} ({delta_tool*100:+.1f} pts)")
print(f"  Δ Arg F1      (Enhanced - Original) : {delta_f1:+.4f}")
print(f"  Δ UI Guide    (Enhanced - Original) : {delta_guide:+.4f}")

## Cell 11 — Per-language breakdown

In [ ]:
lang_rows = []
for label, stats in [("Enhanced", enh_stats), ("Original", orig_stats)]:
    for lang, v in sorted(stats.items()):
        n = v["n"]
        lang_rows.append({
            "system":   label,
            "language": lang.upper(),
            "tool_acc": round(v["ok"] / n, 4) if n else 0,
            "arg_f1":   round(v["f1"] / n, 4) if n else 0,
            "guide_acc":round(v["guide_ok"] / n, 4) if n else 0,
            "p50_ms":   round(sorted(v["lats"])[len(v["lats"]) // 2], 1) if v["lats"] else 0,
            "n":        n,
        })

df_lang = pd.DataFrame(lang_rows)

print("Per-language results:")
print(df_lang.to_string(index=False))

## Cell 12 — Per-tool accuracy breakdown

In [ ]:
def per_tool_accuracy(records: list, label: str) -> dict:
    tool_ok = defaultdict(int)
    tool_n  = defaultdict(int)
    for r in records:
        t = r["gold_tool"]
        tool_n[t]  += 1
        tool_ok[t] += r["tool_ok"]
    return {
        t: {"system": label, "tool": t, "acc": round(tool_ok[t] / tool_n[t], 4), "n": tool_n[t]}
        for t in tool_n
    }


enh_per_tool  = per_tool_accuracy(enh_records,  "Enhanced")
orig_per_tool = per_tool_accuracy(orig_records, "Original")

tool_rows = []
for t in sorted(VALID_TOOLS):
    e = enh_per_tool.get(t, {"acc": 0.0, "n": 0})
    o = orig_per_tool.get(t, {"acc": 0.0, "n": 0})
    tool_rows.append({
        "tool":          t,
        "n":             e.get("n", 0),
        "enhanced_acc":  e.get("acc", 0.0),
        "original_acc":  o.get("acc", 0.0),
        "delta":         round(e.get("acc", 0.0) - o.get("acc", 0.0), 4),
    })

df_tool = pd.DataFrame(tool_rows).sort_values("delta", ascending=False)
print("Per-tool accuracy (Enhanced vs Original):")
print(df_tool.to_string(index=False))

## Cell 13 — Failure analysis: Enhanced Prompt

In [ ]:
print(f"Enhanced prompt failures: {len(enh_fails)} / {len(rows)}")
print()

# Confusion pairs
confusion = defaultdict(int)
for f in enh_fails:
    confusion[(f["gold_tool"], f["pred_tool"])] += 1

print("Confusion pairs (gold → predicted):")
for (gold, pred), cnt in sorted(confusion.items(), key=lambda x: -x[1]):
    print(f"  {gold:<25} → {pred:<25}  count={cnt}")

print()
print("Individual failures (first 20):")
df_fails = pd.DataFrame(enh_fails)[["row_id","language","query","gold_tool","pred_tool","guide_ok","lat_ms"]]
pd.set_option("display.max_colwidth", 60)
print(df_fails.head(20).to_string(index=False))

## Cell 14 — Failure analysis: Original Prompt

In [ ]:
print(f"Original prompt failures: {len(orig_fails)} / {len(rows)}")
print()

confusion_orig = defaultdict(int)
for f in orig_fails:
    confusion_orig[(f["gold_tool"], f["pred_tool"])] += 1

print("Confusion pairs (gold → predicted):")
for (gold, pred), cnt in sorted(confusion_orig.items(), key=lambda x: -x[1]):
    print(f"  {gold:<25} → {pred:<25}  count={cnt}")

## Cell 15 — Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("NexusBots — Enhanced vs Original Prompt  (Qwen3-0.6B Base)",
             fontsize=14, fontweight="bold")

colors = ["#2196F3", "#FF9800"]  # blue=enhanced, orange=original

# ── Plot 1: Overall metrics bar chart ────────────────────────────────────────
ax = axes[0]
metrics     = ["Tool Acc", "Arg F1", "UI Guide Acc"]
enh_vals    = [results[0]["tool_acc"], results[0]["arg_f1"],  results[0]["ui_guide_acc"]]
orig_vals   = [results[1]["tool_acc"], results[1]["arg_f1"],  results[1]["ui_guide_acc"]]

x    = range(len(metrics))
w    = 0.35
bars1 = ax.bar([i - w/2 for i in x], enh_vals,  w, label="Enhanced", color=colors[0], alpha=0.85)
bars2 = ax.bar([i + w/2 for i in x], orig_vals, w, label="Original", color=colors[1], alpha=0.85)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_title("Overall Metrics")
ax.legend()
ax.set_ylabel("Score")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)


# ── Plot 2: Per-language tool accuracy ───────────────────────────────────────
ax = axes[1]
langs_list = sorted(enh_stats.keys())
enh_lang   = [enh_stats[l]["ok"] / enh_stats[l]["n"] for l in langs_list]
orig_lang  = [orig_stats.get(l, {"ok": 0, "n": 1})["ok"] /
               orig_stats.get(l, {"ok": 0, "n": 1})["n"] for l in langs_list]

x2 = range(len(langs_list))
ax.bar([i - w/2 for i in x2], enh_lang,  w, label="Enhanced", color=colors[0], alpha=0.85)
ax.bar([i + w/2 for i in x2], orig_lang, w, label="Original", color=colors[1], alpha=0.85)
ax.set_xticks(list(x2))
ax.set_xticklabels([l.upper() for l in langs_list])
ax.set_ylim(0, 1.1)
ax.set_title("Per-Language Tool Accuracy")
ax.legend()
ax.set_ylabel("Tool Accuracy")

for bars, vals in [(ax.containers[0], enh_lang), (ax.containers[1], orig_lang)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)


# ── Plot 3: Per-tool accuracy delta ──────────────────────────────────────────
ax = axes[2]
df_tool_sorted = df_tool.sort_values("delta", ascending=True)
bar_colors = ["#4CAF50" if d >= 0 else "#F44336" for d in df_tool_sorted["delta"]]
ax.barh(df_tool_sorted["tool"], df_tool_sorted["delta"], color=bar_colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Per-Tool Accuracy Delta\n(Enhanced − Original)")
ax.set_xlabel("Δ Accuracy")
for i, (_, row) in enumerate(df_tool_sorted.iterrows()):
    ax.text(row["delta"] + (0.005 if row["delta"] >= 0 else -0.005),
            i, f"{row['delta']:+.3f}",
            va="center", ha="left" if row["delta"] >= 0 else "right", fontsize=9)

plt.tight_layout()
out_png = RESULTS_DIR / "enhanced_prompt_benchmark.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"Saved → {out_png}")
plt.show()

## Cell 16 — Latency distribution

In [ ]:
enh_lats  = [r["lat_ms"] for r in enh_records  if r["lat_ms"] > 0]
orig_lats = [r["lat_ms"] for r in orig_records if r["lat_ms"] > 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(enh_lats,  bins=20, alpha=0.7, color=colors[0], label="Enhanced",  density=True)
ax.hist(orig_lats, bins=20, alpha=0.7, color=colors[1], label="Original",  density=True)

import statistics
for lats, color, label in [(enh_lats, colors[0], "Enhanced"), (orig_lats, colors[1], "Original")]:
    med = statistics.median(lats)
    ax.axvline(med, color=color, linestyle="--", linewidth=1.5,
               label=f"{label} p50={med:.0f}ms")

ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Density")
ax.set_title("Inference Latency Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "latency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Enhanced  — mean={sum(enh_lats)/len(enh_lats):.0f}ms  "
      f"p50={sorted(enh_lats)[len(enh_lats)//2]:.0f}ms  "
      f"p95={sorted(enh_lats)[int(len(enh_lats)*0.95)]:.0f}ms")
print(f"Original  — mean={sum(orig_lats)/len(orig_lats):.0f}ms  "
      f"p50={sorted(orig_lats)[len(orig_lats)//2]:.0f}ms  "
      f"p95={sorted(orig_lats)[int(len(orig_lats)*0.95)]:.0f}ms")

## Cell 17 — Confusion matrix heatmap

In [ ]:
import numpy as np

TOOLS_ORDERED = sorted(VALID_TOOLS)
tool_idx = {t: i for i, t in enumerate(TOOLS_ORDERED)}

def build_confusion_matrix(records):
    mat = np.zeros((len(TOOLS_ORDERED), len(TOOLS_ORDERED)), dtype=int)
    for r in records:
        g = r["gold_tool"]
        p = r["pred_tool"]
        if g in tool_idx:
            pi = tool_idx.get(p, -1)
            if pi >= 0:
                mat[tool_idx[g]][pi] += 1
    return mat

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix — Gold (rows) vs Predicted (cols)",
             fontsize=13, fontweight="bold")

for ax, records, title in [
    (axes[0], enh_records,  "Enhanced Prompt"),
    (axes[1], orig_records, "Original Prompt"),
]:
    mat = build_confusion_matrix(records)
    im  = ax.imshow(mat, cmap="Blues")
    ax.set_xticks(range(len(TOOLS_ORDERED)))
    ax.set_yticks(range(len(TOOLS_ORDERED)))
    ax.set_xticklabels(TOOLS_ORDERED, rotation=35, ha="right", fontsize=8)
    ax.set_yticklabels(TOOLS_ORDERED, fontsize=8)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Gold")
    ax.set_title(title)
    for i in range(len(TOOLS_ORDERED)):
        for j in range(len(TOOLS_ORDERED)):
            if mat[i, j] > 0:
                color = "white" if mat[i, j] > mat.max() * 0.5 else "black"
                ax.text(j, i, str(mat[i, j]), ha="center", va="center",
                        fontsize=10, color=color, fontweight="bold")
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Cell 18 — Save results to CSV

In [ ]:
import csv

# Overall summary CSV
summary_csv = RESULTS_DIR / "enhanced_prompt_results.csv"
df_results.to_csv(summary_csv, index=False)
print(f"Saved summary     → {summary_csv}")

# Per-row detail CSVs
enh_csv  = RESULTS_DIR / "enhanced_prompt_per_row.csv"
orig_csv = RESULTS_DIR / "original_prompt_per_row.csv"
pd.DataFrame(enh_records).to_csv(enh_csv, index=False)
pd.DataFrame(orig_records).to_csv(orig_csv, index=False)
print(f"Saved enhanced    → {enh_csv}")
print(f"Saved original    → {orig_csv}")

# Per-language CSV
lang_csv = RESULTS_DIR / "per_language_results.csv"
df_lang.to_csv(lang_csv, index=False)
print(f"Saved per-lang    → {lang_csv}")

# Failures CSV
fail_csv = RESULTS_DIR / "enhanced_prompt_failures.csv"
pd.DataFrame(enh_fails).drop(columns=["raw_output"], errors="ignore").to_csv(fail_csv, index=False)
print(f"Saved failures    → {fail_csv}")

## Cell 19 — Final summary printout

In [ ]:
print("\n" + "="*70)
print("  BENCHMARK COMPLETE — NexusBots Enhanced Prompt Evaluation")
print("="*70)

print(f"\n  Model       : {BASE_MODEL}")
print(f"  Test set    : {len(rows)} rows  ({TEST_PATH.name})")
print(f"  Device      : {DEVICE}")

print("\n  ┌─────────────────────────────────────┬──────────┬────────┬──────────┬─────────┐")
print(  "  │ System                              │ Tool Acc │ Arg F1 │ UI Guide │ p50 ms  │")
print(  "  ├─────────────────────────────────────┼──────────┼────────┼──────────┼─────────┤")
for r in results:
    name = r["system"][:36].ljust(36)
    print(f"  │ {name} │  {r['tool_acc']:.4f}  │ {r['arg_f1']:.4f} │  {r['ui_guide_acc']:.4f}  │ {r['p50_ms']:>7.1f} │")
print(  "  └─────────────────────────────────────┴──────────┴────────┴──────────┴─────────┘")

print(f"\n  Improvement (Enhanced − Original):")
print(f"    Tool Accuracy : {delta_tool:+.4f}  ({delta_tool*100:+.1f} percentage points)")
print(f"    Arg F1        : {delta_f1:+.4f}")
print(f"    UI Guide Acc  : {delta_guide:+.4f}")

print(f"\n  Output files saved to: {RESULTS_DIR}")
print("="*70)